# Stage 4b: Student KD Training (Proof of Concept)

**Goal:** Train the pruned student against pre-computed self-play teacher targets.
Verify that distillation improves over the raw pruned baseline.

**Run on:** Colab Free (T4) or Pro (A100)

**Workflow:**
1. Configure variant + loss function
2. Stage data + load pruned model
3. Pre-KD mimicking evaluation (baseline snapshot)
4. Train with knowledge distillation
5. Post-KD mimicking evaluation (comparison table)
6. Export final model

**Outputs saved to:** `eval/{exp_id}/kd/{variant}_{loss}/` and `checkpoints/{exp_id}/kd/{variant}_{loss}/`

## Cell 1: Session Startup

In [1]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA L4
VRAM: 23.7 GB


## Cell 2: Experiment Configuration

All experiment parameters in one place. Change `VARIANT_NAME` and `LOSS_CONFIG`
to run different ablations — all output paths update automatically.

In [2]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT CONFIGURATION — edit these to run different ablations    ║
# ╚══════════════════════════════════════════════════════════════════════╝

EXPERIMENT_ID  = "prune30_v1"                     # Stage 2 experiment ID

# ── Pruned Variants ──
# Depth Strategies:
#   v1_scattered             : Hard layer removal, avoids dual-stream-critical layers
#   v2a_cont_strict    : Contiguous block removal, strictly 0 critical layers
#   v2b_cont_penalized : Contiguous block, penalized for critical layers
#   v2c_cont_relaxed   : Contiguous block, ignores tags (pure BI ranking)
#   v3_collapse              : Layer collapse (residual merge), avoids critical layers
# Width Modifiers (append to depth):
#   _nonuni                  : Non-uniform pruning ratio per layer (SlimLLM style)
#   _uni                     : Uniform pruning across all chosen layers
# Example: "v1_scattered_nonuni" or "v3_collapse_uni"
VARIANT_NAME   = "sparsegpt_30pct"

# ── Loss Configurations ──
# L1 : Logit KD only (Soft matches teacher token distribution)
# L2 : Logit KD + Hard Label (CE on teacher's chosen tokens) -> Default/Recommended
# L3 : Logit KD + Hard Label + Hidden State Alignment (Needs large disk space for targets)
# L4 : Logit KD + Hard Label + Depth Codebook Cross-Entropy (Targets CB 1-7)
# L5 : All combined (Logit KD + Hard Label + Hidden States + Codebook CE)
LOSS_CONFIG    = "L2"

# ── Training hyperparameters ──
LR             = 1e-4
BATCH_SIZE     = 4
GRAD_ACCUM     = 4       # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
NUM_EPOCHS     = 3
ALPHA          = 0.7     # soft logit KD weight
DELTA          = 0.3     # hard label CE weight
TEMPERATURE    = 3.0     # KD temperature


# ── Derived paths (all keyed by VARIANT_NAME + LOSS_CONFIG) ──
RUN_ID = f"{VARIANT_NAME}_{LOSS_CONFIG}"

GDRIVE_BASE    = "/content/drive/MyDrive/moshilite"
GDRIVE_DATA    = f"{GDRIVE_BASE}/self_play_data/conversations"
LOCAL_DATA     = "/content/staged/self_play"
WEIGHTS_DIR    = f"{GDRIVE_BASE}/upstream_weights/moshiko"
PRUNED_WEIGHTS = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}/{VARIANT_NAME}.pt"

CHECKPOINT_DIR = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}/kd/{RUN_ID}"
EVAL_DIR       = f"{GDRIVE_BASE}/eval/{EXPERIMENT_ID}/kd/{RUN_ID}"
EXPORT_PATH    = f"{GDRIVE_BASE}/models/{RUN_ID}_kd.pt"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print(f"Experiment:  {EXPERIMENT_ID}")
print(f"Variant:     {VARIANT_NAME}")
print(f"Loss config: {LOSS_CONFIG}")
print(f"Run ID:      {RUN_ID}")
print(f"")
print(f"Pruned weights: {PRUNED_WEIGHTS}")
print(f"Checkpoints:    {CHECKPOINT_DIR}")
print(f"Eval output:    {EVAL_DIR}")
print(f"Export path:    {EXPORT_PATH}")

Experiment:  prune30_v1
Variant:     sparsegpt_30pct
Loss config: L2
Run ID:      sparsegpt_30pct_L2

Pruned weights: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/sparsegpt_30pct.pt
Checkpoints:    /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/kd/sparsegpt_30pct_L2
Eval output:    /content/drive/MyDrive/moshilite/eval/prune30_v1/kd/sparsegpt_30pct_L2
Export path:    /content/drive/MyDrive/moshilite/models/sparsegpt_30pct_L2_kd.pt


## Cell 3: Stage Self-Play Data to Local Disk

In [5]:
import shutil
from pathlib import Path

# Copy self-play data to local NVMe for fast training reads
src = Path(GDRIVE_DATA)
dst = Path(LOCAL_DATA)

if dst.exists():
    print(f"Local data already staged at {dst}")
else:
    print(f"Staging data from GDrive to local disk...")
    shutil.copytree(str(src), str(dst))

npz_count = len(list(dst.rglob("*.npz")))
total_mb = sum(f.stat().st_size for f in dst.rglob("*.npz")) / 1e6
print(f"Staged: {npz_count} conversations, {total_mb:.0f} MB")

Staging data from GDrive to local disk...
Staged: 480 conversations, 4 MB


## Cell 4: Load Pruned Student Model

In [6]:
import torch, json, shutil
import torch.nn as nn
from pathlib import Path
from moshi.models import loaders

gpu_name = torch.cuda.get_device_name(0).lower()
DTYPE = torch.bfloat16 if any(x in gpu_name for x in ["a100", "h100", "l4"]) else torch.float16
print(f"Using {DTYPE} on {gpu_name}")

print(f"Loading pruned student: {PRUNED_WEIGHTS}")
weights_path = Path(WEIGHTS_DIR)
config = None
moshi_name = "model.safetensors"

if weights_path.exists():
    config_path = weights_path / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)
        moshi_name = config.get("moshi_name", moshi_name)

# Step 1: Stage + load the pruned checkpoint
local_pruned = Path("/content/staged_pruned.pt")
local_pruned.unlink(missing_ok=True)
print("Staging pruned weights to local disk...")
shutil.copy2(PRUNED_WEIGHTS, str(local_pruned))
pruned_ckpt = torch.load(str(local_pruned), map_location="cpu", weights_only=False)
local_pruned.unlink()
pruned_state = pruned_ckpt["model_state_dict"]
print(f"  Variant: {pruned_ckpt.get('variant_name', 'unknown')}")
print(f"  Params:  {pruned_ckpt.get('param_billions', '?')}B")

# Step 2: Build the base model architecture from upstream weights
base_weights_file = weights_path / moshi_name
if not base_weights_file.exists():
    sf_files = [f for f in weights_path.glob("*.safetensors")
                if "tokenizer" not in f.name.lower()]
    base_weights_file = sf_files[0] if sf_files else None

print(f"  Loading base architecture from: {base_weights_file}")
student = loaders.get_moshi_lm(
    str(base_weights_file), lm_kwargs=config, device="cpu", dtype=DTYPE
)

# Step 3: Match layer count — depth pruning removed layers and renumbered
pruned_layer_idxs = sorted({
    int(k.split(".")[2]) for k in pruned_state
    if k.startswith("transformer.layers.")
})
n_pruned_layers = max(pruned_layer_idxs) + 1
base_n_layers = len(student.transformer.layers)
print(f"  Adjusting layers: {base_n_layers} -> {n_pruned_layers}")

while len(student.transformer.layers) > n_pruned_layers:
    del student.transformer.layers[-1]

# Step 4: Force-assign all parameters/buffers from pruned state dict
assigned, skipped = 0, 0
for name, tensor in pruned_state.items():
    parts = name.split(".")
    try:
        module = student
        for part in parts[:-1]:
            module = getattr(module, part)
        attr_name = parts[-1]
        old = getattr(module, attr_name, None)
        if isinstance(old, nn.Parameter):
            setattr(module, attr_name, nn.Parameter(tensor, requires_grad=old.requires_grad))
        else:
            setattr(module, attr_name, tensor)
        assigned += 1
    except Exception as e:
        skipped += 1
        print(f"  ⚠️ Could not assign {name}: {e}")

print(f"  ✅ Assigned {assigned} tensors, skipped {skipped}")
del pruned_ckpt, pruned_state

student = student.to("cuda")
student.train()

# Freeze early layers — only train last 8
n_layers = len(student.transformer.layers)
for i, layer in enumerate(student.transformer.layers):
    if i < n_layers - 8:
        for p in layer.parameters():
            p.requires_grad = False
print(f"Frozen first {n_layers - 8}/{n_layers} layers")

if hasattr(student.transformer, "set_checkpointing"):
    student.transformer.set_checkpointing(True)
    print("Gradient checkpointing enabled")

n_params = sum(p.numel() for p in student.parameters()) / 1e9
n_trainable = sum(p.numel() for p in student.parameters() if p.requires_grad) / 1e9
print(f"Student loaded: {n_params:.2f}B params ({n_trainable:.2f}B trainable)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


Using torch.bfloat16 on nvidia l4
Loading pruned student: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/sparsegpt_30pct.pt
Staging pruned weights to local disk...


KeyboardInterrupt: 

## Cell 4b: Pre-KD Mimicking Evaluation

Evaluate the pruned model **before** distillation training.
This gives a baseline snapshot to measure how much KD improves the student.

**Time:** ~30s on A100, ~1 min on T4

In [5]:
import time, json
from moshilite.distillation.evaluator import MimickingEvaluator
from moshilite.data.dataset import get_self_play_dataloader

# Create evaluator (same hyperparams as training)
evaluator = MimickingEvaluator(
    alpha=ALPHA, delta=DELTA, temperature=TEMPERATURE,
    device="cuda", use_amp=True,
)

# Val loader (same deterministic split used during training)
val_loader = get_self_play_dataloader(
    LOCAL_DATA, split="val", batch_size=BATCH_SIZE, val_fraction=0.1,
)

print("\n" + "="*68)
print(f"  PRE-KD MIMICKING EVALUATION  [{VARIANT_NAME}]")
print("="*68)
student.eval()
t0 = time.perf_counter()
pre_kd_results = evaluator.evaluate(student, val_loader)
t1 = time.perf_counter()
student.train()

print(f"  Completed in {t1 - t0:.1f}s\n")
for k, v in pre_kd_results.items():
    print(f"    {k}: {v}")

# Save pre-KD eval
pre_kd_path = os.path.join(EVAL_DIR, "pre_kd_eval.json")
with open(pre_kd_path, "w") as f:
    json.dump(pre_kd_results, f, indent=2)
print(f"\nSaved to {pre_kd_path}")

# Quick health check
if pre_kd_results.get("text_perplexity", 0) > 10000:
    print("\n" + "!"*60)
    print("  WARNING: Text perplexity is extremely high (>10,000).")
    print("  This pruned variant may be too damaged for KD to recover.")
    print("  Consider trying a less aggressive variant.")
    print("!"*60)

📂 SelfPlayDataset [val]: 48 conversations from /content/staged/self_play

  PRE-KD MIMICKING EVALUATION  [sparsegpt_30pct]


/content/moshilite/src/moshilite/distillation/evaluator.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=amp_dtype, enabled=self.use_amp):
W0417 21:24:17.032000 13395 torch/_inductor/utils.py:1613] [2/0] Not enough SMs to use max_autotune_gemm mode
/content/moshilite/src/moshilite/distillation/evaluator.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=amp_dtype, enabled=self.use_amp):


  Completed in 37.4s

    text_token_acc: 0.0634
    audio_cb0_token_acc: 0.0001
    text_top5_agree: 0.8883
    audio_cb0_top5_agree: 0.0003
    text_kl_div: nan
    audio_cb0_kl_div: nan
    text_perplexity: nan
    audio_cb0_perplexity: nan
    val_loss_l2: nan

Saved to /content/drive/MyDrive/moshilite/eval/prune30_v1/kd/sparsegpt_30pct_L2/pre_kd_eval.json


## Cell 5: Run Training

In [5]:
from moshilite.distillation.trainer import StudentTrainer

trainer = StudentTrainer(
    student_model=student,
    data_dir=LOCAL_DATA,
    checkpoint_dir=CHECKPOINT_DIR,
    lr=LR,
    batch_size=BATCH_SIZE,
    gradient_accumulation=GRAD_ACCUM,
    alpha=ALPHA,
    delta=DELTA,
    temperature=TEMPERATURE,
    checkpoint_every=200,
    val_every=50,        # validate more often since fewer steps
    device="cuda",
    use_amp=True,
)

summary = trainer.train(num_epochs=NUM_EPOCHS)


📂 SelfPlayDataset [train]: 432 conversations from /content/staged/self_play
📂 SelfPlayDataset [val]: 48 conversations from /content/staged/self_play
✅ Using 8-bit AdamW (bitsandbytes)
🔧 AMP dtype: torch.bfloat16, GradScaler: OFF

🚀 Training: 3 epochs × 27 steps/epoch = 81 total steps


Epoch 2/3:  85%|████████▌ | 92/108 [00:42<00:39,  2.45s/it, step=50, loss=nan]


📊 Val loss @ step 50: nan


Epoch 3/3: 100%|██████████| 108/108 [00:41<00:00,  2.58it/s, step=81, loss=nan]


💾 Checkpoint saved: checkpoint_latest.pt (step 81)
🧹 Cleaned up local staging dir (/content/ckpt_staging)

✅ Training complete: 81 steps, 3.5 min, best val loss: inf
📝 Summary saved to /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/kd/sparsegpt_30pct_L2/training_summary.json


## Cell 6: Load Best Checkpoint & Training Summary

In [6]:
# Best weights already restored by trainer — no disk load needed
student.eval()

print(f"\nTraining Summary [{RUN_ID}]:")
print(f"   Total steps:     {summary['total_steps']}")
print(f"   Best val loss:   {summary['best_val_loss']:.4f}")
print(f"   Final train loss:{summary['final_train_loss']:.4f}")
print(f"   Training time:   {summary['elapsed_seconds'] / 60:.1f} min")



Training Summary [sparsegpt_30pct_L2]:
   Total steps:     81
   Best val loss:   inf
   Final train loss:nan
   Training time:   3.5 min


## Cell 6b: Post-KD Mimicking Evaluation

Evaluate the distilled student on the same held-out self-play validation split
and compare against the pre-KD baseline from Cell 4b.

**Time:** ~30s on A100, ~1 min on T4

In [7]:
import time, json

print("\n" + "="*68)
print(f"  POST-KD MIMICKING EVALUATION  [{RUN_ID}]")
print("="*68)
t0 = time.perf_counter()
post_kd_results = evaluator.evaluate(student, val_loader)
t1 = time.perf_counter()
print(f"  Completed in {t1 - t0:.1f}s\n")
for k, v in post_kd_results.items():
    print(f"    {k}: {v}")

# Print comparison table: Post-KD vs Pre-KD
evaluator.print_comparison(post_kd_results, pre_kd_results)

# Save all evaluation results
eval_output = {
    "run_id": RUN_ID,
    "variant_name": VARIANT_NAME,
    "loss_config": LOSS_CONFIG,
    "hyperparameters": {
        "lr": LR, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "alpha": ALPHA, "delta": DELTA, "temperature": TEMPERATURE,
        "num_epochs": NUM_EPOCHS,
    },
    "pre_kd": pre_kd_results,
    "post_kd": post_kd_results,
    "training_summary": summary,
}

# Save post-KD eval
post_kd_path = os.path.join(EVAL_DIR, "post_kd_eval.json")
with open(post_kd_path, "w") as f:
    json.dump({"post_kd": post_kd_results}, f, indent=2, default=str)

# Save combined results
combined_path = os.path.join(EVAL_DIR, "full_eval_results.json")
with open(combined_path, "w") as f:
    json.dump(eval_output, f, indent=2, default=str)

# Save training summary separately
summary_path = os.path.join(EVAL_DIR, "training_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nAll results saved to {EVAL_DIR}/")
print(f"  - pre_kd_eval.json")
print(f"  - post_kd_eval.json")
print(f"  - full_eval_results.json")
print(f"  - training_summary.json")


  POST-KD MIMICKING EVALUATION  [sparsegpt_30pct_L2]


NameError: name 'evaluator' is not defined

## Cell 7: Export Student Model

In [ ]:
# Save the distilled student as a standalone model
from pathlib import Path

torch.save({
    "model_state_dict": student.state_dict(),
    "config": {
        "run_id": RUN_ID,
        "variant_name": VARIANT_NAME,
        "loss_config": LOSS_CONFIG,
        "best_val_loss": summary["best_val_loss"],
        "training_steps": summary["total_steps"],
        "num_epochs": NUM_EPOCHS,
        "alpha": ALPHA,
        "delta": DELTA,
        "temperature": TEMPERATURE,
    },
    "pre_kd_eval": pre_kd_results,
    "post_kd_eval": post_kd_results,
}, EXPORT_PATH)

# Update model manifest
try:
    from moshilite.utils.experiment import update_model_manifest
    update_model_manifest(
        model_filename=f"{RUN_ID}_kd.pt",
        source_checkpoint=str(best_ckpt),
        experiment_id=EXPERIMENT_ID,
        metrics={
            "best_val_loss": summary["best_val_loss"],
            "training_steps": summary["total_steps"],
            **post_kd_results,
        },
    )
except Exception as e:
    print(f"Manifest update failed (non-critical): {e}")

export_size = Path(EXPORT_PATH).stat().st_size / 1e9
print(f"Student exported to {EXPORT_PATH}")
print(f"Size: {export_size:.2f} GB")

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display

# Assuming BASE_EVAL_DIR is defined above (e.g., "/content/drive/MyDrive/moshilite/eval/prune30_v1")
kd_dir = os.path.join(BASE_EVAL_DIR, "kd")
kd_records = []

if os.path.exists(kd_dir):
    for run_folder in os.listdir(kd_dir):
        run_path = os.path.join(kd_dir, run_folder)
        if not os.path.isdir(run_path): continue

        record = {"Model Variant": run_folder}

        # 1. Load Training Summary
        summary_path = os.path.join(run_path, "training_summary.json")
        if os.path.exists(summary_path):
            with open(summary_path, 'r') as f:
                summary = json.load(f)
            record["Total steps"] = summary.get("total_steps", 0)
            record["Best val loss"] = f"{summary.get('best_val_loss', 0):.4f}"
            record["Final train loss"] = f"{summary.get('final_train_loss', 0):.4f}"
            record["Training time (min)"] = f"{summary.get('elapsed_seconds', 0) / 60.0:.1f}"

        # 2. Load Post-KD Mimicking Evaluation
        post_eval_path = os.path.join(run_path, "post_kd_eval.json")
        if os.path.exists(post_eval_path):
            with open(post_eval_path, 'r') as f:
                post_eval = json.load(f)

            # Extract nested dict if needed
            if "post_kd" in post_eval:
                post_eval = post_eval["post_kd"]

            record["text_token_acc"] = f"{post_eval.get('text_token_acc', 0.0):.4f}"
            record["audio_cb0_token_acc"] = f"{post_eval.get('audio_cb0_token_acc', 0.0):.4f}"
            record["text_top5_agree"] = f"{post_eval.get('text_top5_agree', 0.0):.4f}"
            record["audio_cb0_top5_agree"] = f"{post_eval.get('audio_cb0_top5_agree', 0.0):.4f}"
            record["text_kl_div"] = f"{post_eval.get('text_kl_div', 0.0):.4f}"
            record["audio_cb0_kl_div"] = f"{post_eval.get('audio_cb0_kl_div', 0.0):.4f}"
            record["text_perplexity"] = f"{post_eval.get('text_perplexity', 0.0):.2f}"
            record["audio_cb0_perplexity"] = f"{post_eval.get('audio_cb0_perplexity', 0.0):.2f}"
            record["val_loss_l2"] = f"{post_eval.get('val_loss_l2', 0.0):.4f}"

        if len(record) > 1:
            kd_records.append(record)

    df_kd_summary = pd.DataFrame(kd_records).set_index("Model Variant")
    print("--- Training Summary & Post-KD Results ---")
    display(df_kd_summary)
else:
    print(f"Directory not found: {kd_dir}")


In [ ]:
import os
import json
import pandas as pd
from IPython.display import display

kd_dir = os.path.join(BASE_EVAL_DIR, "kd")
compare_records = []

metrics_to_compare = [
    "text_token_acc", "audio_cb0_token_acc",
    "text_top5_agree", "audio_cb0_top5_agree",
    "text_kl_div", "audio_cb0_kl_div",
    "text_perplexity", "audio_cb0_perplexity",
    "val_loss_l2"
]

if os.path.exists(kd_dir):
    for run_folder in os.listdir(kd_dir):
        run_path = os.path.join(kd_dir, run_folder)
        if not os.path.isdir(run_path): continue

        record = {"Model Variant": run_folder}

        # Load Pre-KD (Pruned Baseline)
        pre_eval_path = os.path.join(run_path, "pre_kd_eval.json")
        pre_eval = {}
        if os.path.exists(pre_eval_path):
            with open(pre_eval_path, 'r') as f:
                pre_eval = json.load(f)

        # Load Post-KD (Distilled Model)
        post_eval_path = os.path.join(run_path, "post_kd_eval.json")
        post_eval = {}
        if os.path.exists(post_eval_path):
            with open(post_eval_path, 'r') as f:
                post_eval = json.load(f)
            if "post_kd" in post_eval:
                post_eval = post_eval["post_kd"]

        # Map values side by side
        for metric in metrics_to_compare:
            pre_val = pre_eval.get(metric, None)
            post_val = post_eval.get(metric, None)

            # Format precision logically depending on metric
            if "perplexity" in metric:
                record[(metric, "Pre-KD")] = f"{pre_val:.2f}" if pre_val is not None else "N/A"
                record[(metric, "Post-KD")] = f"{post_val:.2f}" if post_val is not None else "N/A"
            else:
                record[(metric, "Pre-KD")] = f"{pre_val:.4f}" if pre_val is not None else "N/A"
                record[(metric, "Post-KD")] = f"{post_val:.4f}" if post_val is not None else "N/A"

        if len(record) > 1:
            compare_records.append(record)

    if compare_records:
        df_compare = pd.DataFrame(compare_records)
        df_compare.set_index("Model Variant", inplace=True)

        # Initialize MultiIndex columns for a grouped tabular design
        df_compare.columns = pd.MultiIndex.from_tuples(df_compare.columns)

        print("--- Pre-KD vs Post-KD Distillation Comparison ---")
        display(df_compare)
else:
    print(f"Directory not found: {kd_dir}")


NameError: name 'BASE_EVAL_DIR' is not defined